In [0]:
print("Spark version:", spark.version)
print("Databricks is ready!")

In [0]:
import os
print("current working directory",os.getcwd())
print("\nFiles and folders")
print(os.listdir())

In [0]:
file_path = "file:/Workspace/Repos/Git/education-analytics-lakehouse/data/sample/universities.csv"

print(file_path)

In [0]:
universities_df = (
    spark.read
    .option("header",True)
    .option("inferSchema",True)
    .csv(file_path)
)
universities_df.show()

In [0]:
print(universities_df.columns)
universities_df.printSchema()
print("Total Records",universities_df.count())

In [0]:
from pyspark.sql.functions import current_timestamp,lit
# 1. Add audit metadata columns for data lineage (without altering raw data values)
bronze_universities_df = (
    universities_df
    .withColumn("ingestion_timestamp",current_timestamp())
)

In [0]:
from pyspark.sql.functions import current_timestamp, lit

# 1. Add audit metadata columns for data lineage
bronze_universities_df = (
    universities_df
    .withColumn("ingestion_timestamp", current_timestamp())
    .withColumn("source_file", lit(file_path))
)

# 2. Save as a managed Delta table in the default database/metastore
# This lets Databricks handle the storage path automatically behind the scenes!
(
    bronze_universities_df
    .write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("bronze_universities")
)

print("Bronze table successfully saved as a managed Delta table: bronze_universities")

# 3. Verify by reading it back from the table name
verified_bronze_df = spark.read.table("bronze_universities")
verified_bronze_df.show(5, truncate=False)
print("Total records in Bronze Delta table:", verified_bronze_df.count())